# Task 8: Evaluator and Model Selection

This task evaluates the October Legal-BERT models, compares weighted and unweighted training, and compares the selected model with the September baselines.

## Step 1: Build and Validate the Evaluation Data

Load Tasks 3, 6, and 7 and confirm that the validation chunks, categories, labels, masks, and transformer scores are aligned.

In [ ]:
#Load Task 3 and Task 7 validation data

import pandas as pd
import numpy as np

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/"
    "Accenture-1O-contract-review-challenge/"
    "main/notebooks/"
)

# Task 3
task3_val = pd.read_parquet(
    BASE_URL
    + "october_task3_outputs_11_2/"
    + "task3_labels_mask_validation.parquet"
)

# Task 7
baseline_rows = pd.read_parquet(
    BASE_URL
    + "october_task7_outputs/"
    + "task7_baseline_row_level_predictions.parquet"
)

# Keep validation rows only
baseline_val = baseline_rows[
    baseline_rows["split"] == "validation"
].copy()

print("Task 3 shape:", task3_val.shape)
print("Task 7 validation shape:", baseline_val.shape)

Task 3 shape: (2873, 3)
Task 7 validation shape: (28730, 9)


In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np
import json

drive.mount("/content/drive")

TASK4_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/october_task4_outputs"
)

print("Folder exists:", TASK4_OUTPUT_DIR.exists())
print("Contents:", [p.name for p in TASK4_OUTPUT_DIR.iterdir()])

Mounted at /content/drive
Folder exists: True
Contents: ['results_unweighted.json']


In [ ]:
# Download the shared Task 6 output folder into Colab

!pip -q install gdown

import gdown

FOLDER_URL = "https://drive.google.com/drive/folders/1uICW6WAehxzo2Pqr7oB_dAxoE03pw1zp?usp=drive_link"

gdown.download_folder(
    FOLDER_URL,
    output="/content/october_task4_outputs",
    quiet=False,
    use_cookies=False
)



Retrieving folder contents


Retrieving folder 1MqOFSacaqC-bFtyHBYemocuD_Msmf4_d full_unweighted
Retrieving folder 1C32FWOn7vWBCprgqVGoytzymBR8L1UyE .ipynb_checkpoints
Retrieving folder 17RibpEVpqiijBtU4paNPyhZO5qfaogQe final_unweighted_Legal-BERT_seed42
Processing file 12Ca98IYP5nQQqHnAbLXESLJgX4vfdKcw config.json
Processing file 1YS_uGLtMzraskdHnCtIg7HKZXg3146Nt model.safetensors
Retrieving folder 1U9v3fQ4o5qREja4Dj9zZz2-J_NaVYo_i final_unweighted_Legal-BERT_seed123
Processing file 1dpSHMtXb0nj1pcinfzslqNz2sAw7mmM5 config.json
Processing file 1HcN5By0USjFCGhodN7Kc47e3wsjW4uqz model.safetensors
Retrieving folder 1yVI3AQBCuZL8_EOa-XkmOP3EIwTv2Ylx final_unweighted_Legal-BERT_seed2026
Processing file 13Yzzb3IeZImS9nZx-w3z-7v5kcNtFZRO config.json
Processing file 1BVcmUCz1Rom1ePTh3RjhTg_dJ3Ax3mXw model.safetensors
Retrieving folder 1pRwqTs7gtmIqhe0s_x_rleHEhUVzz774 run_unweighted_Legal-BERT_seed42
Retrieving folder 1_Qy0uVsWH_r7vq4twooxEzn0OjLT2gSP checkpoint-2034
Processing file 1DET2PJjc2VNoQzeiUwNzYYUlynQPfQFX conf

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=12Ca98IYP5nQQqHnAbLXESLJgX4vfdKcw
To: /content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed42/config.json
100%|██████████| 1.48k/1.48k [00:00<00:00, 2.53MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1YS_uGLtMzraskdHnCtIg7HKZXg3146Nt
From (redirected): https://drive.google.com/uc?id=1YS_uGLtMzraskdHnCtIg7HKZXg3146Nt&confirm=t&uuid=93e55f34-5c15-4edc-a0b6-8de058c810e2
To: /content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed42/model.safetensors
100%|██████████| 438M/438M [00:02<00:00, 200MB/s]
Downloading...
From: https://drive.google.com/uc?id=1dpSHMtXb0nj1pcinfzslqNz2sAw7mmM5
To: /content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed123/config.json
100%|██████████| 1.48k/1.48k [00:00<00:00, 4.50MB/s]
Downloading...
From (original): h

['/content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed42/config.json',
 '/content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed42/model.safetensors',
 '/content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed123/config.json',
 '/content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed123/model.safetensors',
 '/content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed2026/config.json',
 '/content/october_task4_outputs/full_unweighted/final_unweighted_Legal-BERT_seed2026/model.safetensors',
 '/content/october_task4_outputs/full_unweighted/run_unweighted_Legal-BERT_seed42/checkpoint-2034/config.json',
 '/content/october_task4_outputs/full_unweighted/run_unweighted_Legal-BERT_seed42/checkpoint-2034/model.safetensors',
 '/content/october_task4_outputs/full_unweighted/run_unweighted_Legal-BERT_seed42/checkpoint-2034/tokenizer_config.json',
 '/content/october_task4_outputs/ful

In [ ]:
from pathlib import Path

TASK4_OUTPUT_DIR = Path("/content/october_task4_outputs")

print("Folder exists:", TASK4_OUTPUT_DIR.exists())

for path in TASK4_OUTPUT_DIR.iterdir():
    print(path.name)

Folder exists: True
train_chunk_ids.json
full_unweighted
full_weighted
val_chunk_ids.json


In [ ]:
import numpy as np
import json

SEEDS = [42, 123, 2026]

def load_run(run, seed):
    folder = TASK4_OUTPUT_DIR / f"full_{run}"

    out = np.load(
        folder / f"{run}_Legal-BERT_seed{seed}_val_outputs.npz",
        allow_pickle=True
    )

    scores = np.load(
        folder / f"{run}_Legal-BERT_seed{seed}_val_scores.npz",
        allow_pickle=True
    )

    return (
        {k: out[k] for k in out.files},
        {k: scores[k] for k in scores.files}
    )

runs = {
    (run, seed): load_run(run, seed)
    for run in ("unweighted", "weighted")
    for seed in SEEDS
}

with open(TASK4_OUTPUT_DIR / "val_chunk_ids.json", "r") as f:
    val_ids = json.load(f)

print("Runs loaded:", len(runs))
print("Validation chunks:", len(val_ids))

Runs loaded: 6
Validation chunks: 2873


In [ ]:
#Basic alignment checks

print("Task 3 chunks:", task3_val["chunk_id"].nunique())
print("Task 7 chunks:", baseline_val["chunk_id"].nunique())
print("Categories:", baseline_val["category_id"].nunique())

print(
    "Same chunk IDs:",
    set(task3_val["chunk_id"]) == set(baseline_val["chunk_id"])
)

print(
    "Duplicate pairs:",
    baseline_val.duplicated(
        ["chunk_id", "category_id"]
    ).sum()
)

Task 3 chunks: 2873
Task 7 chunks: 2873
Categories: 10
Same chunk IDs: True
Duplicate pairs: 0


In [ ]:
#Check labels, masks, and usable evaluation rows

print("Mask counts:")
print(baseline_val["mask"].value_counts())

eval_df = baseline_val[
    baseline_val["mask"] == 1
].copy()

print("\nEligible pairs:", len(eval_df))
print("Positive labels:", (eval_df["label"] == 1).sum())
print("Negative labels:", (eval_df["label"] == 0).sum())

Mask counts:
mask
1    28690
0       40
Name: count, dtype: int64

Eligible pairs: 28690
Positive labels: 563
Negative labels: 28127


In [ ]:
#Convert Task 6 logits to probabilities

SEEDS = [42, 123, 2026]

FINAL_IDS = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity"
]

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

transformer_tables = {}

for run in ("unweighted", "weighted"):
    for seed in SEEDS:

        _, scores = runs[(run, seed)]

        probs = sigmoid(scores["val_logits"])

        transformer_tables[(run, seed)] = pd.DataFrame({
            "chunk_id": np.repeat(scores["chunk_ids"], len(FINAL_IDS)),
            "category_id": FINAL_IDS * len(scores["chunk_ids"]),
            "score": probs.reshape(-1)
        })

print(
    transformer_tables[("unweighted", 42)].shape
)

(28730, 3)


In [ ]:
#Add transformer scores and final check

evaluation_table = baseline_val.copy()

for run in ("unweighted", "weighted"):
    for seed in SEEDS:

        temp = transformer_tables[(run, seed)].rename(
            columns={
                "score": f"{run}_seed{seed}_score"
            }
        )

        evaluation_table = evaluation_table.merge(
            temp,
            on=["chunk_id", "category_id"],
            how="left"
        )

score_cols = [
    f"{run}_seed{seed}_score"
    for run in ("unweighted", "weighted")
    for seed in SEEDS
]

print("Rows:", len(evaluation_table))
print("Missing scores:", evaluation_table[score_cols].isna().sum().sum())
print("Unique chunks:", evaluation_table["chunk_id"].nunique())
print("Unique categories:", evaluation_table["category_id"].nunique())

Rows: 28730
Missing scores: 0
Unique chunks: 2873
Unique categories: 10


### Step 1 Conclusion

The validation data was successfully aligned across Tasks 3, 6, and 7. All 2,873 validation chunks and 10 categories matched, with no duplicate pairs or missing transformer scores. After masking, 28,690 chunk-category pairs remained for evaluation.

## Step 2: Thresholds and Model Evaluation

Use only unmasked validation pairs, select the best per-category thresholds, and calculate precision, recall, and F1 for weighted and unweighted Legal-BERT.

In [ ]:
#Keep only unmasked validation pairs

from sklearn.metrics import precision_score, recall_score, f1_score

eval_unmasked = evaluation_table[
    evaluation_table["mask"] == 1
].copy()

print("Evaluation rows:", len(eval_unmasked))
print("Positive labels:", eval_unmasked["label"].sum())
print("Negative labels:", (eval_unmasked["label"] == 0).sum())

Evaluation rows: 28690
Positive labels: 563
Negative labels: 28127


In [ ]:
#Find the best threshold using validation F1

THRESHOLDS = np.arange(0.05, 1.00, 0.05)

def find_best_threshold(y_true, scores):
    best_threshold = 0.50
    best_f1 = -1

    for threshold in THRESHOLDS:
        predictions = (scores >= threshold).astype(int)

        score = f1_score(
            y_true,
            predictions,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    return best_threshold

In [ ]:
#Evaluate all transformer runs

metric_rows = []

for run in ("unweighted", "weighted"):
    for seed in SEEDS:

        score_col = f"{run}_seed{seed}_score"

        for category in FINAL_IDS:

            data = eval_unmasked[
                eval_unmasked["category_id"] == category
            ]

            y_true = data["label"].to_numpy()
            scores = data[score_col].to_numpy()

            threshold = find_best_threshold(
                y_true,
                scores
            )

            predictions = (
                scores >= threshold
            ).astype(int)

            metric_rows.append({
                "run": run,
                "seed": seed,
                "category": category,
                "threshold": threshold,
                "precision": precision_score(
                    y_true, predictions, zero_division=0
                ),
                "recall": recall_score(
                    y_true, predictions, zero_division=0
                ),
                "f1": f1_score(
                    y_true, predictions, zero_division=0
                )
            })

metrics_df = pd.DataFrame(metric_rows)

metrics_df.head(10)

,run,seed,category,threshold,precision,recall,f1
0,unweighted,42,governing_law,0.55,0.975904,0.952941,0.964286
1,unweighted,42,renewal_term,0.35,0.758621,0.758621,0.758621
2,unweighted,42,revenue_profit_sharing,0.40,0.677419,0.525000,0.591549
3,unweighted,42,cap_on_liability,0.75,0.950820,0.852941,0.899225
4,unweighted,42,uncapped_liability,0.35,0.484848,0.727273,0.581818
5,unweighted,42,termination_for_convenience,0.30,0.500000,0.676471,0.575000
6,unweighted,42,anti_assignment,0.55,0.910256,0.835294,0.871166
7,unweighted,42,audit_rights,0.70,0.910448,0.835616,0.871429
8,unweighted,42,license_grant,0.45,0.886076,0.813953,0.848485
9,unweighted,42,exclusivity,0.40,0.700000,0.341463,0.459016


In [ ]:
#Average performance across seeds

summary_df = (
    metrics_df
    .groupby(["run", "category"])
    .agg(
        mean_threshold=("threshold", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std")
    )
    .reset_index()
)

summary_df.round(3)

,run,category,mean_threshold,mean_precision,mean_recall,mean_f1,std_f1
0,unweighted,anti_assignment,0.333,0.879,0.875,0.876,0.011
1,unweighted,audit_rights,0.650,0.939,0.826,0.879,0.007
2,unweighted,cap_on_liability,0.533,0.909,0.868,0.887,0.011
3,unweighted,exclusivity,0.367,0.687,0.512,0.574,0.135
4,unweighted,governing_law,0.300,0.958,0.973,0.965,0.001
5,unweighted,license_grant,0.517,0.879,0.818,0.847,0.012
6,unweighted,renewal_term,0.417,0.767,0.759,0.763,0.028
7,unweighted,revenue_profit_sharing,0.400,0.670,0.608,0.632,0.048
8,unweighted,termination_for_convenience,0.367,0.559,0.735,0.627,0.051
9,unweighted,uncapped_liability,0.367,0.476,0.758,0.584,0.024


In [ ]:
macro_summary = (
    summary_df
    .groupby("run")
    [["mean_precision", "mean_recall", "mean_f1"]]
    .mean()
    .round(3)
)

macro_summary

,mean_precision,mean_recall,mean_f1
run,,,
unweighted,0.772,0.773,0.764
weighted,0.759,0.809,0.770


In [ ]:
#Class imbalance effect table

imbalance_table = (
    eval_unmasked
    .groupby("category_id")["label"]
    .agg(
        total="count",
        positives="sum"
    )
    .reset_index()
)

imbalance_table["negatives"] = (
    imbalance_table["total"]
    - imbalance_table["positives"]
)

imbalance_table["positive_rate"] = (
    imbalance_table["positives"]
    / imbalance_table["total"]
)

# Add average F1 for weighted and unweighted models
f1_compare = summary_df.pivot(
    index="category",
    columns="run",
    values="mean_f1"
).reset_index()

# Match the column name before merging
f1_compare = f1_compare.rename(
    columns={"category": "category_id"}
)

imbalance_table = imbalance_table.merge(
    f1_compare,
    on="category_id"
)

imbalance_table["weighted_f1_change"] = (
    imbalance_table["weighted"]
    - imbalance_table["unweighted"]
)

imbalance_table.round(3)

,category_id,total,positives,negatives,positive_rate,unweighted,weighted,weighted_f1_change
0,anti_assignment,2872,85,2787,0.030,0.876,0.883,0.006
1,audit_rights,2870,73,2797,0.025,0.879,0.857,-0.021
2,cap_on_liability,2869,68,2801,0.024,0.887,0.879,-0.008
3,exclusivity,2866,41,2825,0.014,0.574,0.668,0.093
4,governing_law,2873,85,2788,0.030,0.965,0.961,-0.004
5,license_grant,2860,86,2774,0.030,0.847,0.817,-0.030
6,renewal_term,2869,29,2840,0.010,0.763,0.743,-0.020
7,revenue_profit_sharing,2867,40,2827,0.014,0.632,0.680,0.048
8,termination_for_convenience,2872,34,2838,0.012,0.627,0.686,0.059
9,uncapped_liability,2872,22,2850,0.008,0.584,0.521,-0.063


### Step 2 Conclusion

Weighted Legal-BERT achieved a macro F1 of 0.770 compared with 0.764 for the unweighted model. Weighting increased recall from 0.773 to 0.809, although its effect varied across categories. This showed that class weighting helped overall performance on the imbalanced validation data.

## Step 3: Baseline Comparison and Selection

Compare Legal-BERT with the September keyword and logistic regression baselines, then select the final model family.

In [ ]:
#Evaluate September baselines on the same unmasked validation pairs

baseline_metric_rows = []

for model, pred_col in [
    ("logistic_regression", "lr_prediction"),
    ("keyword", "keyword_prediction")
]:
    for category in FINAL_IDS:

        data = eval_unmasked[
            eval_unmasked["category_id"] == category
        ]

        y_true = data["label"].to_numpy()
        y_pred = data[pred_col].to_numpy()

        baseline_metric_rows.append({
            "model": model,
            "category": category,
            "precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "f1": f1_score(
                y_true, y_pred, zero_division=0
            )
        })

baseline_metrics = pd.DataFrame(baseline_metric_rows)

baseline_metrics.round(3)

,model,category,precision,recall,f1
0,logistic_regression,governing_law,0.602,0.765,0.674
1,logistic_regression,renewal_term,0.545,0.621,0.581
2,logistic_regression,revenue_profit_sharing,0.444,0.600,0.511
3,logistic_regression,cap_on_liability,0.429,0.618,0.506
4,logistic_regression,uncapped_liability,0.478,0.500,0.489
5,logistic_regression,termination_for_convenience,0.333,0.353,0.343
6,logistic_regression,anti_assignment,0.576,0.671,0.620
7,logistic_regression,audit_rights,0.696,0.753,0.724
8,logistic_regression,license_grant,0.571,0.651,0.609
9,logistic_regression,exclusivity,0.522,0.293,0.375


In [ ]:
#Compare overall macro performance

baseline_macro = (
    baseline_metrics
    .groupby("model")[["precision", "recall", "f1"]]
    .mean()
    .reset_index()
)

transformer_macro = pd.DataFrame({
    "model": ["Legal-BERT unweighted", "Legal-BERT weighted"],
    "precision": [
        macro_summary.loc["unweighted", "mean_precision"],
        macro_summary.loc["weighted", "mean_precision"]
    ],
    "recall": [
        macro_summary.loc["unweighted", "mean_recall"],
        macro_summary.loc["weighted", "mean_recall"]
    ],
    "f1": [
        macro_summary.loc["unweighted", "mean_f1"],
        macro_summary.loc["weighted", "mean_f1"]
    ]
})

overall_comparison = pd.concat(
    [baseline_macro, transformer_macro],
    ignore_index=True
)

overall_comparison.round(3)

,model,precision,recall,f1
0,keyword,0.644,0.208,0.257
1,logistic_regression,0.520,0.582,0.543
2,Legal-BERT unweighted,0.772,0.773,0.764
3,Legal-BERT weighted,0.759,0.809,0.770


In [ ]:
#Per-category model comparison

weighted_f1 = (
    summary_df[
        summary_df["run"] == "weighted"
    ][["category", "mean_f1"]]
    .rename(columns={"mean_f1": "weighted_legalbert_f1"})
)

baseline_f1 = (
    baseline_metrics
    .pivot(
        index="category",
        columns="model",
        values="f1"
    )
    .reset_index()
)

category_comparison = weighted_f1.merge(
    baseline_f1,
    on="category"
)

category_comparison["vs_lr"] = (
    category_comparison["weighted_legalbert_f1"]
    - category_comparison["logistic_regression"]
)

category_comparison["vs_keyword"] = (
    category_comparison["weighted_legalbert_f1"]
    - category_comparison["keyword"]
)

category_comparison.round(3)

,category,weighted_legalbert_f1,keyword,logistic_regression,vs_lr,vs_keyword
0,anti_assignment,0.883,0.023,0.620,0.263,0.860
1,audit_rights,0.857,0.523,0.724,0.134,0.334
2,cap_on_liability,0.879,0.141,0.506,0.373,0.738
3,exclusivity,0.668,0.093,0.375,0.293,0.575
4,governing_law,0.961,0.468,0.674,0.288,0.493
5,license_grant,0.817,0.516,0.609,0.209,0.301
6,renewal_term,0.743,0.067,0.581,0.162,0.676
7,revenue_profit_sharing,0.680,0.136,0.511,0.169,0.544
8,termination_for_convenience,0.686,0.227,0.343,0.343,0.459
9,uncapped_liability,0.521,0.378,0.489,0.032,0.143


In [ ]:
#Final selected model family and frozen thresholds

selected_run = "weighted"

final_thresholds = (
    metrics_df[
        metrics_df["run"] == selected_run
    ][
        ["seed", "category", "threshold",
         "precision", "recall", "f1"]
    ]
    .sort_values(["category", "seed"])
    .reset_index(drop=True)
)

print("Selected model family: Weighted Legal-BERT")
print("Seeds retained:", SEEDS)
print("Number of threshold records:", len(final_thresholds))

final_thresholds.round(3)

Selected model family: Weighted Legal-BERT
Seeds retained: [42, 123, 2026]
Number of threshold records: 30


,seed,category,threshold,precision,recall,f1
0,42,anti_assignment,0.90,0.901,0.859,0.880
1,123,anti_assignment,0.75,0.892,0.871,0.881
2,2026,anti_assignment,0.90,0.893,0.882,0.888
3,42,audit_rights,0.95,0.909,0.822,0.863
4,123,audit_rights,0.75,0.825,0.904,0.863
5,2026,audit_rights,0.90,0.829,0.863,0.846
6,42,cap_on_liability,0.75,0.840,0.926,0.881
7,123,cap_on_liability,0.85,0.910,0.897,0.904
8,2026,cap_on_liability,0.95,0.902,0.809,0.853
9,42,exclusivity,0.95,0.781,0.610,0.685


Based on these results, I would record the selection as:


Selected model family: Weighted Legal-BERT. Across the three validation seeds, weighted Legal-BERT achieved a macro precision of 0.759, macro recall of 0.809, and macro F1 of 0.770. This exceeded unweighted Legal-BERT's macro F1 of 0.764, logistic regression's 0.543, and the keyword baseline's 0.257. Weighted Legal-BERT also outperformed logistic regression and the keyword baseline in F1 across all 10 contract categories.

For the imbalance finding:


Class weighting increased overall recall from 0.773 to 0.809 and slightly improved macro F1 from 0.764 to 0.770. However, its effect varied by category: it substantially helped categories such as exclusivity and termination-for-convenience, while reducing F1 for categories such as uncapped liability and license grant.

### Step 3 Conclusion

Weighted Legal-BERT outperformed both September baselines, with a macro F1 of 0.770 compared with 0.543 for logistic regression and 0.257 for the keyword baseline. It also achieved higher F1 scores than both baselines across all 10 categories, so weighted Legal-BERT was selected as the final model family.

## Final Selection and Outputs

Weighted Legal-BERT is selected because it achieved the highest macro F1 and recall. The final thresholds, evaluation metrics, comparisons, and selection decision are saved as frozen Task 8 outputs.

In [ ]:
#Freeze and download Task 8 outputs

from pathlib import Path
import json
from google.colab import files

# 1. Create output folder

OUTPUT_DIR = Path("october_task8_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


# 2. Freeze final Task 8 tables

# Per-seed, per-category thresholds
final_thresholds.to_csv(
    OUTPUT_DIR / "task8_frozen_thresholds.csv",
    index=False
)

# Overall model comparison
overall_comparison.to_csv(
    OUTPUT_DIR / "task8_model_comparison.csv",
    index=False
)

# Per-category comparison against baselines
category_comparison.to_csv(
    OUTPUT_DIR / "task8_category_comparison.csv",
    index=False
)

# Class imbalance effect table
imbalance_table.to_csv(
    OUTPUT_DIR / "task8_imbalance_effect.csv",
    index=False
)

# Full transformer metric results
metrics_df.to_csv(
    OUTPUT_DIR / "task8_transformer_metrics.csv",
    index=False
)

# Baseline metric results
baseline_metrics.to_csv(
    OUTPUT_DIR / "task8_baseline_metrics.csv",
    index=False
)


# 3. Freeze final selection decision

task8_selection = {
    "selected_model_family": "weighted Legal-BERT",
    "seeds_retained": [42, 123, 2026],
    "evaluation_split": "validation",
    "evaluation_mask": 1,
    "eligible_evaluation_pairs": 28690,
    "positive_pairs": 563,
    "negative_pairs": 28127,

    "weighted_legalbert": {
        "macro_precision": 0.759,
        "macro_recall": 0.809,
        "macro_f1": 0.770
    },

    "unweighted_legalbert": {
        "macro_precision": 0.772,
        "macro_recall": 0.773,
        "macro_f1": 0.764
    },

    "logistic_regression_macro_f1": 0.543,
    "keyword_macro_f1": 0.257,

    "threshold_rule": {
        "selection_split": "validation only",
        "range": "0.05 to 0.95",
        "step": 0.05,
        "criterion": "maximum per-category F1",
        "note": "Seed-specific thresholds retained; mean thresholds are not frozen."
    }
}

with open(
    OUTPUT_DIR / "task8_frozen_selection.json",
    "w"
) as f:
    json.dump(task8_selection, f, indent=2)

# 4. Zip all Task 8 outputs

import shutil

shutil.make_archive(
    "october_task8_outputs",
    "zip",
    OUTPUT_DIR
)

print("Task 8 outputs frozen successfully.")
print("\nSaved files:")

for file in OUTPUT_DIR.iterdir():
    print("-", file.name)


# 5. Download ZIP file

files.download("october_task8_outputs.zip")

Task 8 outputs frozen successfully.

Saved files:
- task8_model_comparison.csv
- task8_imbalance_effect.csv
- task8_transformer_metrics.csv
- task8_frozen_thresholds.csv
- task8_baseline_metrics.csv
- task8_frozen_selection.json
- task8_category_comparison.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Final Task 8 Conclusion

Task 8 completed the evaluation and model selection process. Weighted Legal-BERT was selected, all three seeds were retained, and the per-category thresholds, evaluation metrics, baseline comparisons, and imbalance results were frozen for later use.